# Supervised Learning — Part 3: Classification Algorithms (00:00 – 28:44)

- **Course:** Sheryians AI School — Complete Machine Learning Course for Beginners (Instructor: Akarsh Vyas)
- **Duration Covered:** `00:00 – 28:44`
- **Resume Bookmark:** `28:44` *(currently inside Linear Regression vs. Logistic Regression)*
- **Companion Visual PDF:** `supervised_learning_part3_28_mins.pdf`
- **Synced Gemini Notebook:** `supervised learning part 2notebook`

---

## Video Progress Map
| Timestamp | Chapter | Status |
| :--- | :--- | :--- |
| `00:00 – 05:11` | Intro & Part 3 Roadmap | Completed |
| `05:11 – 08:32` | What is Classification? | Completed |
| `08:32 – 27:27` | Logistic Regression, Sigmoid, Odds & Decision Boundary | Completed |
| `27:27 – 28:44` | Linear Regression vs. Logistic Regression | **Paused at 28:44** |
| `31:21 – 45:11` | Log Loss (Binary Cross-Entropy) | Next |
| `45:11 – 56:24+` | Scikit-Learn Code, KNN, Trees, Naive Bayes, SVM & Titanic | Upcoming |

## 1. What is Classification? (`05:11 – 08:32`)

In **Part 2 (Regression)**, we predicted continuous numeric values (e.g., exam score, house price).  
In **Part 3 (Classification)**, we predict **discrete class labels**:
- **Binary Classification (2 classes):** Spam (`1`) vs. Not Spam (`0`), Pass vs. Fail, Survived vs. Not Survived.
- **Multiclass Classification (3+ classes):** Cat / Dog / Bird, Handwritten Digits (`0–9`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression

# Sigmoid (Logistic) function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Worked numerical table for key z values
print(f"{'z':>6} | {'e^(-z)':>10} | {'sigmoid(z)':>12} | {'Class (thr=0.5)':>15}")
print("-" * 52)
for z in [-4, -2, -1, 0, 1, 2, 4]:
    p = sigmoid(z)
    cls = 1 if p >= 0.5 else 0
    print(f"{z:>6} | {np.exp(-z):>10.4f} | {p:>12.4f} | {cls:>15}")

## 2. The Sigmoid Curve (`08:32 – 20:00`)

Logistic Regression computes a linear score $z = w_1 x_1 + \dots + w_n x_n + b$ and passes it through the Sigmoid function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

This squashes any real number $(-\infty, +\infty)$ into a probability strictly between `0` and `1`.

In [ ]:
z_vals = np.linspace(-6, 6, 300)
probs = sigmoid(z_vals)

plt.figure(figsize=(8, 4))
plt.plot(z_vals, probs, color='#4F46E5', linewidth=2.5, label=r'$\sigma(z) = 1 / (1 + e^{-z})$')
plt.axhline(0.5, color='#D97706', linestyle='--', label='Decision Threshold (P = 0.5)')
plt.axvline(0.0, color='gray', linestyle=':', alpha=0.7)
plt.scatter([-4, 0, 4], [sigmoid(-4), sigmoid(0), sigmoid(4)], color=['#E11D48', '#D97706', '#0D9488'], zorder=5)
plt.title('Sigmoid (Logistic) Function')
plt.xlabel('Linear Score z = w·x + b')
plt.ylabel('Probability P(y = 1 | x)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.show()

## 3. Odds & Log-Odds (Logit) (`20:00 – 27:27`)

- **Odds:** $\text{Odds} = \frac{p}{1 - p} \in [0, +\infty)$
- **Log-Odds (Logit):** $\ln\left(\frac{p}{1 - p}\right) = w \cdot x + b = z \in (-\infty, +\infty)$

Solving the Log-Odds equation for $p$ yields the exact Sigmoid formula!

In [ ]:
# Explore Probability -> Odds -> Log-Odds
for p in [0.1, 0.25, 0.5, 0.75, 0.8, 0.9]:
    odds = p / (1 - p)
    logit = np.log(odds)
    print(f"p = {p:.2f}  ->  Odds = {odds:5.2f}  ->  Log-Odds (z) = {logit:+6.3f}")

## 4. Linear Regression vs. Logistic Regression (`27:27 – 28:44`, Paused Here)

Below we reproduce the exact failure mode of using `LinearRegression` for binary classification when a single extreme outlier is added, compared with `LogisticRegression`.

In [ ]:
X_base = np.array([1.0, 1.5, 2.0, 2.4, 2.8, 4.2, 4.7, 5.2, 5.7, 6.1]).reshape(-1, 1)
y_base = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

X_with_outlier = np.vstack([X_base, [[14.5]]])
y_with_outlier = np.append(y_base, 1)

lin_clean = LinearRegression().fit(X_base, y_base)
lin_outlier = LinearRegression().fit(X_with_outlier, y_with_outlier)
log_outlier = LogisticRegression().fit(X_with_outlier, y_with_outlier)

x_grid = np.linspace(0.5, 15.0, 200).reshape(-1, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.scatter(X_base, y_base, c=y_base, cmap='bwr_r', s=55, edgecolors='k')
ax1.scatter([14.5], [1], color='gold', s=120, marker='*', edgecolors='k', label='Outlier')
ax1.plot(x_grid, lin_clean.predict(x_grid), '--', color='#4F46E5', label='Linear (No Outlier)')
ax1.plot(x_grid, lin_outlier.predict(x_grid), '-', color='#E11D48', linewidth=2, label='Linear (With Outlier)')
ax1.axhline(0.5, color='gray', linestyle=':')
ax1.set_ylim(-0.25, 1.45)
ax1.set_title('Linear Regression Fails (Tilted + Unbounded)')
ax1.legend()

ax2.scatter(X_base, y_base, c=y_base, cmap='bwr_r', s=55, edgecolors='k')
ax2.scatter([14.5], [1], color='gold', s=120, marker='*', edgecolors='k', label='Outlier')
ax2.plot(x_grid, log_outlier.predict_proba(x_grid)[:, 1], color='#0D9488', linewidth=2.5, label='Logistic Sigmoid')
ax2.axhline(0.5, color='gray', linestyle=':')
ax2.set_ylim(-0.25, 1.45)
ax2.set_title('Logistic Regression Stays Bounded in (0, 1)')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Next Steps (Starting at `28:44`)
1. **`28:44 – 31:21`** — Finish Linear Regression vs. Logistic Regression comparison.
2. **`31:21 – 45:11`** — Study the **Log Loss (Binary Cross-Entropy)** cost function.
3. **`45:11 – 56:24`** — Code `LogisticRegression` with `scikit-learn` (reusing `pd.get_dummies`, `train_test_split`, `StandardScaler`).
4. **`56:24+`** — KNN, Decision Trees, Naive Bayes, SVM, Evaluation Metrics (Confusion Matrix, Precision, Recall, F1), and the **Titanic Survival Prediction** project.